# MeChess 3: an opening explorer from the Lichess database (CPU notebook)

**Run with the accelerator OFF (None).** This is streaming and counting: CPU only, no GPU quota.

It reads a Lichess monthly database dump (CC0) as a stream (the 30 GB file is never downloaded whole), counts every position and move of the first 16 plies for five rating
bands (600-1000, 1000-1400, 1400-1800, 1800-2200, 2200+), and writes:

| File | What it is |
|---|---|
| `explorer.db` | SQLite opening explorer: per rating band and position, each move's games and White / draw / Black results. Query it with `chessme explorer-query` |
| `theory_LO_HI.bin` | one **playable opening book** per band (the format the engine already reads; `mechess --book <folder>` picks the band of the target Elo) |
| `report.md` | how deep each book carries games it has never seen (held-out games), per band |

**How to run**
1. *Settings -> Internet -> On*, *Accelerator -> None*. Put your repository URL in `REPO_URL`.
2. **Save Version -> Save & Run All (Commit)**.
3. Expect roughly 1 to 3 hours with the defaults (20 million games scanned; one in four is looked at and about half of those qualify, so about 2.5 million games are counted; peak memory about 4 GB). If it stops early, the time budget still writes all outputs from what was counted.
4. Download the output. The books go in `data/book/` (the explorer, wherever you like).

**Safety.** Step 1 (`--check`) reads a few hundred games first: it fails in seconds if the URL, the decompression or the parser is broken. The counting is resumable: a checkpoint is
written every million games, and if this notebook's earlier output is added as an input, the run continues from it. The stream reconnects itself if the connection drops.

Sampling: games in a dump are ordered by time, so the notebook scans a large window and keeps every 4th game instead of the first few million. Bullet games and games that ended abnormally are excluded, and both players must
be within 300 rating points of each other. 2000 games per band are held out to measure coverage honestly.

In [ ]:
import os, subprocess, sys, pathlib, glob, time

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here
MONTH = "2026-06"                                       # a month on https://database.lichess.org (standard rated)
MAX_SCAN = 20_000_000                                   # games to scan
SAMPLE_EVERY = 4                                        # count one game in this many
MAX_MINUTES = 600                                       # time budget; outputs are written even if it is reached
SOURCE = f"https://database.lichess.org/standard/lichess_db_standard_rated_{MONTH}.pgn.zst"
OUT = "/kaggle/working/explorer" if os.path.exists("/kaggle") else "explorer_out"
T0 = time.time()

def sh(*args):
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    if "<you>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your repository first.")
    sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]

os.makedirs(OUT, exist_ok=True)
prev = sorted(glob.glob("/kaggle/input/**/explorer/state.pkl.gz", recursive=True))      # an earlier run of this notebook, added as an input
if prev and not os.path.exists(f"{OUT}/state.pkl.gz"):
    subprocess.run(["cp", prev[0], f"{OUT}/state.pkl.gz"], check=True)
    print("resuming from", prev[0])
print("source:", SOURCE)

## Step 1. Check: read a few hundred games (seconds). Stops here if the URL, decompression or parser is broken.

In [ ]:
sh(*CLI, "explorer-build", "--check", "--source", SOURCE)

## Step 2. Count and build (the long step; resumable)

In [ ]:
remaining = max(5, MAX_MINUTES - (time.time() - T0) / 60)
sh(*CLI, "explorer-build", "--source", SOURCE, "--out", OUT, "--max-scan", str(MAX_SCAN), "--sample-every", str(SAMPLE_EVERY),
   "--max-minutes", str(round(remaining, 1)), "--checkpoint-every", "1000000", "--log", f"{OUT}/explorer.log")
print(f"elapsed {(time.time() - T0) / 60:.1f} min")

## Step 3. Try the explorer

In [ ]:
START = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
AFTER_E4 = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1"
for rating in (1000, 1500, 2000):
    print(f"--- rating {rating}, after 1.e4")
    sh(*CLI, "explorer-query", f"{OUT}/explorer.db", "--rating", str(rating), "--fen", AFTER_E4, "--top", "5")

In [ ]:
# Keep the output small: the resume checkpoint is only needed if the run did not finish
import json
log = open(f"{OUT}/explorer.log").read()
finished = "'finished': True" in log
ck = pathlib.Path(OUT, "state.pkl.gz")
if finished and ck.exists():
    ck.unlink()
print("finished:", finished, "| output:", sorted(os.listdir(OUT)), "|", round(sum(f.stat().st_size for f in pathlib.Path(OUT).rglob('*') if f.is_file()) / 1e6), "MB")